[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C59_VLA_Perception_Interface_Course/00_setup/00_environment_check.ipynb)

# 00 · 课程总览与环境（三代架构信息流 / 长尾覆盖率 / 感知接口初见）

目标：在**同一个场景**上，把「模块化流水线 / 端到端 / VLA」三种架构各跑一遍，
看清它们分别在哪一步丢掉了信息、以及为什么只有第三种能处理「前方施工改道」。

本 notebook 你会亲手实现：
1. 一个可复现的合成场景（含**置信度、距离、车道关联、临时/固定来源**四类信息）
2. **模块化流水线**：规则表 + 置信度阈值 + 冲突消解 —— 并看它在长尾场景上怎么输错
3. **端到端**：最近邻策略 —— 并量化「它错了但你查不出为什么」
4. **VLA 风格**：感知输出序列化成文本 + 自然语言先验 —— 并看它为什么能对
5. 三种架构的**信息流量化对比**（可解释中间量个数 / 延迟 / 长尾覆盖）
6. **Zipf 覆盖率模型**：算出「50% 要 53 条规则，99% 要 4562 条」
7. ✏️ 三道练习 + 📖 参考答案 + 🧪 TSR→VLA 接口契约草案

> 心智模型：**规则法的成本随覆盖率超线性上升，学习法次线性上升——
> 两条曲线一定会交叉，交叉点之后就该换范式。**

## 1 · 环境自检

本课**只需要 numpy**。没有 torch、没有 GPU、不联网。

In [ ]:
import sys, math, json, collections
import numpy as np

print('python  :', sys.version.split()[0])
print('numpy   :', np.__version__)
assert sys.version_info >= (3, 8), '需要 Python 3.8+'
assert int(np.__version__.split('.')[0]) >= 1, 'numpy 版本异常'

# 本课刻意不依赖任何深度学习框架：机制要手写才学得到
for mod in ['torch', 'tensorflow', 'transformers']:
    print(f'  {mod:<14s} 需要? 否（本课不使用）')

rng = np.random.default_rng(59)   # 全课固定种子，保证可复现
print()
print('✅ 环境就绪：纯 numpy + 标准库，CPU 可跑，不联网')

## 2 · 一个场景：前方施工改道

这是本 notebook 的**唯一场景**，三种架构都吃它。刻意设计成一个
「人类零思考、规则系统必错」的例子。

四类信息缺一不可：
- `score` —— 临时牌又小又脏，置信度天然低（0.71），而固定牌又大又清晰（0.93）
- `dist_m` —— 临时牌更远（82 m），固定牌更近（40 m）→ **「取最近的」这条规则会选错**
- `lane` —— 「车道封闭」说的是右车道，不是自车道
- `src` —— 临时 / 固定，**这是解决冲突的关键，但规则表里往往没有这一维**

In [ ]:
SCENE = {
    'id': 'construction_detour',
    'human_desc': '前方 80 米施工，右侧车道封闭，临时限速 40，锥桶引导向左并道',
    'ego_speed_kmh': 78.0,
    'dets': [
        # cls,                    score, dist_m, lane,    src
        {'cls': 'speed_limit_80',    'score': 0.93, 'dist_m': 40.0, 'lane': 'ego',   'src': 'permanent'},
        {'cls': 'speed_limit_40',    'score': 0.71, 'dist_m': 82.0, 'lane': 'ego',   'src': 'temporary'},
        {'cls': 'construction_ahead','score': 0.63, 'dist_m': 85.0, 'lane': 'ego',   'src': 'temporary'},
        {'cls': 'lane_closed_right', 'score': 0.44, 'dist_m': 88.0, 'lane': 'right', 'src': 'temporary'},
    ],
    # 人类司机（也是 ground truth）会怎么做
    'gt': {'v_target_kmh': 40.0, 'lane_cmd': 'keep_left'},
}

print(f"场景: {SCENE['human_desc']}")
print(f"自车速度: {SCENE['ego_speed_kmh']:.0f} km/h")
print()
print(f"{'类别':<22s}{'score':>7s}{'dist_m':>9s}{'lane':>8s}{'src':>12s}")
for d in SCENE['dets']:
    print(f"{d['cls']:<22s}{d['score']:>7.2f}{d['dist_m']:>9.1f}{d['lane']:>8s}{d['src']:>12s}")
print()
print(f"人类司机（ground truth）: 目标速度 {SCENE['gt']['v_target_kmh']:.0f} km/h, "
      f"车道指令 {SCENE['gt']['lane_cmd']}")
assert len(SCENE['dets']) == 4
print()
print('⚠️  注意两个陷阱：① 固定的 80 牌比临时的 40 牌**更近也更自信**')
print('               ② lane_closed_right 的 score 只有 0.44（被锥桶挡了一半）')

## 3 · 架构 ①：模块化流水线

规则表（人手写）→ 置信度阈值过滤 → 类别查表 → 冲突消解（取最近）。
每一步都合理、都可单测、都「符合设计」。

In [ ]:
# 人手写的规则表：类别 -> 约束。**它只可能包含写规则的人想到的类别。**
RULES = {
    'speed_limit_30':  {'v_max_kmh': 30.0},
    'speed_limit_40':  {'v_max_kmh': 40.0},
    'speed_limit_60':  {'v_max_kmh': 60.0},
    'speed_limit_80':  {'v_max_kmh': 80.0},
    'speed_limit_120': {'v_max_kmh': 120.0},
    'stop':            {'must_stop': True},
    'no_left_turn':    {'ban_maneuver': 'left'},
}
SCORE_TH = 0.60          # 规则层的置信度门限：低于此的检测直接丢弃

def modular_plan(scene, rules=RULES, score_th=SCORE_TH):
    '''模块化流水线：阈值过滤 -> 查表 -> 取最近的限速牌。'''
    trace = {'dropped_lowscore': [], 'unknown_class': [], 'wrong_lane': [], 'fired': []}
    candidates = []
    for d in scene['dets']:
        if d['score'] < score_th:
            trace['dropped_lowscore'].append(d['cls']); continue
        if d['lane'] != 'ego':                    # 只看自车道的标志
            trace['wrong_lane'].append(d['cls']); continue
        if d['cls'] not in rules:                 # 规则表里没有 -> 这条信息消失
            trace['unknown_class'].append(d['cls']); continue
        trace['fired'].append(d['cls'])
        candidates.append((d, rules[d['cls']]))
    # 冲突消解：规则表里**没有**"临时优先于固定"，只能用通用启发式"取最近的"
    speed_c = [(d, c) for d, c in candidates if 'v_max_kmh' in c]
    v = None
    if speed_c:
        d_near, c_near = min(speed_c, key=lambda t: t[0]['dist_m'])
        v = c_near['v_max_kmh']
    return {'v_target_kmh': v, 'lane_cmd': 'keep'}, trace

out1, trace1 = modular_plan(SCENE)
print('中间量（可检查，这是模块化的最大优点）:')
for k, v in trace1.items():
    print(f'  {k:<20s} {v}')
print()
print('输出:', out1)
print('真值:', SCENE['gt'])

assert out1['v_target_kmh'] == 80.0, '取最近的限速牌 -> 选中固定的 80'
assert out1['lane_cmd'] == 'keep'
assert 'construction_ahead' in trace1['unknown_class']
assert 'lane_closed_right' in trace1['dropped_lowscore']
print()
print('❌ 输出 80 km/h + 保持车道 —— 危险，而且**完全符合设计**。')
print('   三处信息丢失: 未知类别 construction_ahead / 低分 lane_closed_right / 无临时优先规则')
print('✅ 但注意: 上面的 trace 有 4 个可检查的中间量 —— **失败是可定位的**。')

## 4 · 架构 ②：端到端

把场景压成一个特征向量，用训练集里最近的样本的动作作为输出。
这是端到端「从数据中学映射」的最小可执行模型——**关键性质是它没有可解释的中间量**。

In [ ]:
VOCAB = ['speed_limit_30','speed_limit_40','speed_limit_60','speed_limit_80',
         'speed_limit_120','stop','no_left_turn','construction_ahead','lane_closed_right']

def scene_feature(scene):
    '''把场景压成固定长度向量：每个类别取最大置信度（这就是"隐向量"的类比）。'''
    f = np.zeros(len(VOCAB) + 1)
    for d in scene['dets']:
        if d['cls'] in VOCAB:
            i = VOCAB.index(d['cls'])
            f[i] = max(f[i], d['score'])
    f[-1] = scene['ego_speed_kmh'] / 120.0        # 归一化自车速度
    return f

def mk(id_, dets, v, lane, ego=78.0):
    return {'id': id_, 'human_desc': id_, 'ego_speed_kmh': ego, 'dets': dets,
            'gt': {'v_target_kmh': v, 'lane_cmd': lane}}

# 训练分布：常见场景。**刻意不含施工改道**——这就是"长尾"的含义
TRAIN = [
    mk('normal_highway', [{'cls':'speed_limit_80','score':0.95,'dist_m':50,'lane':'ego','src':'permanent'}], 80.0, 'keep'),
    mk('urban_stop',     [{'cls':'stop','score':0.91,'dist_m':25,'lane':'ego','src':'permanent'}], 0.0, 'keep', ego=40.0),
    mk('school_zone',    [{'cls':'speed_limit_30','score':0.88,'dist_m':60,'lane':'ego','src':'permanent'}], 30.0, 'keep', ego=45.0),
    mk('highway_fast',   [{'cls':'speed_limit_120','score':0.96,'dist_m':70,'lane':'ego','src':'permanent'}], 120.0, 'keep', ego=110.0),
]
TRAIN_F = np.stack([scene_feature(s) for s in TRAIN])

def e2e_plan(scene):
    '''端到端：一个前向，输出动作。中间没有任何人类可读的量。'''
    f = scene_feature(scene)
    d = np.linalg.norm(TRAIN_F - f, axis=1)
    j = int(np.argmin(d))
    return ({'v_target_kmh': TRAIN[j]['gt']['v_target_kmh'],
             'lane_cmd': TRAIN[j]['gt']['lane_cmd']},
            {'ood_distance': float(d[j]), 'nearest_train_scene': TRAIN[j]['id']})

out2, dbg2 = e2e_plan(SCENE)
print('输出:', out2)
print('真值:', SCENE['gt'])
print()
print(f"最近训练场景: {dbg2['nearest_train_scene']}  距离 {dbg2['ood_distance']:.3f}")
d_in = [float(np.linalg.norm(TRAIN_F - scene_feature(s), axis=1).min()) for s in TRAIN]
print(f"训练集内样本的最近距离: {max(d_in):.3f}  <- 分布内基本为 0")

assert out2['v_target_kmh'] == 80.0, '最近邻落到 normal_highway'
assert dbg2['ood_distance'] > 0.5, '施工场景明显在分布外'
print()
print('❌ 同样输出 80 km/h。而且这一次：')
print('   · 没有 trace、没有中间量，**只有输入和输出**')
print('   · 模型不会说"我不确定"，它照样给出一个自信的数')
print('   · 想修它，只能"再采一批施工场景数据重训"——而长尾有几千种')
print('✅ ood_distance 是唯一的线索，但它只告诉你"怪"，不告诉你"哪里怪"。')

## 5 · 架构 ③：VLA 风格 —— 把感知输出序列化成文本 + 注入自然语言先验

**重要声明**：下面不是真的跑一个大模型。它模拟的是 VLA 的一个关键性质——
**约束来自可组合的自然语言先验，而不是逐类别硬编码的规则表**。
真实 VLA 里这一步由预训练权重完成；这里用关键词组合显式写出来，
好处是你能看清「哪条先验在起作用」。

In [ ]:
def serialize_perception(scene):
    '''① 把结构化检测结果序列化成文本行（m03 会把这个 schema 做深）。'''
    lines = [f"ego_speed: {scene['ego_speed_kmh']:.0f} km/h"]
    for d in sorted(scene['dets'], key=lambda x: x['dist_m']):
        lines.append(
            f"sign: cls={d['cls']} conf={d['score']:.2f} dist={d['dist_m']:.0f}m "
            f"lane={d['lane']} source={d['src']}")
    return lines

# ② 自然语言先验：**注意它们不提任何具体类别**，所以对没见过的标志同样成立
PRIORS = [
    'P1 临时标志的优先级高于固定标志。',
    'P2 出现施工/作业相关标志时，按施工区处理：降速并准备变道。',
    'P3 置信度低于 0.6 的信息不可直接执行，但可作为**保守化**的理由。',
    'P4 车道封闭时，向未封闭的一侧并道。',
    'P5 存在多个速度约束时，取最严格的那个。',
]

PROMPT = serialize_perception(SCENE) + [''] + PRIORS
print('\n'.join(PROMPT))
print()
print(f'prompt 行数 {len(PROMPT)}, 字符数 {sum(len(l) for l in PROMPT)}')
assert any('source=temporary' in l for l in PROMPT), '来源字段必须进 prompt'
assert any('conf=0.44' in l for l in PROMPT), '低置信检测也要进 prompt（由下游决定怎么用）'
print()
print('✅ 关键差别: **低置信的 lane_closed_right 没有被丢弃，而是带着 conf 传下去了**。')
print('   模块化在阈值那一步就把它删了；VLA 让下游自己决定怎么用这个 0.44。')

In [ ]:
def vla_plan(scene, priors=PRIORS):
    '''③ 语义推理（用关键词组合显式模拟"预训练常识"）。
       注意：**没有一条分支写死了具体类别名**，靠的是 source/conf/lane 这些语义维度。'''
    reasoning, constraints = [], []
    speed_signs = [d for d in scene['dets'] if d['cls'].startswith('speed_limit')
                   and d['lane'] == 'ego']
    temp = [d for d in speed_signs if d['src'] == 'temporary']
    perm = [d for d in speed_signs if d['src'] == 'permanent']
    if temp and perm:                                     # P1
        v = min(float(d['cls'].split('_')[-1]) for d in temp)
        reasoning.append(f'P1: 同时存在临时({v:.0f})与固定({perm[0]["cls"]})限速 -> 取临时')
        constraints.append(('v_max_kmh', v))
    elif speed_signs:                                     # P5
        v = min(float(d['cls'].split('_')[-1]) for d in speed_signs)
        reasoning.append(f'P5: 取最严格的速度约束 {v:.0f}')
        constraints.append(('v_max_kmh', v))

    if any('construction' in d['cls'] or 'work' in d['cls'] for d in scene['dets']):
        reasoning.append('P2: 检出施工相关标志 -> 按施工区处理（降速 + 准备变道）')

    lane_cmd = 'keep'
    closed = [d for d in scene['dets'] if 'closed' in d['cls']]
    for d in closed:
        if d['score'] >= 0.60:                            # P4
            lane_cmd = 'keep_left' if d['lane'] == 'right' else 'keep_right'
            reasoning.append(f'P4: {d["lane"]} 车道封闭(conf={d["score"]:.2f}) -> 向另一侧并道')
        else:                                             # P3 低置信 -> 保守
            lane_cmd = 'keep_left' if d['lane'] == 'right' else 'keep_right'
            reasoning.append(f'P3: {d["lane"]} 车道可能封闭(conf={d["score"]:.2f}，低置信) '
                             f'-> 不执行强动作，但**保守地远离**该侧')
    v_target = min([c[1] for c in constraints], default=scene['ego_speed_kmh'])
    return ({'v_target_kmh': v_target, 'lane_cmd': lane_cmd},
            {'reasoning': reasoning, 'constraints': constraints})

out3, dbg3 = vla_plan(SCENE)
print('推理链（可读、可审计——这是 VLA 相比端到端的真正优势）:')
for r in dbg3['reasoning']:
    print('  ·', r)
print()
print('输出:', out3)
print('真值:', SCENE['gt'])

assert out3['v_target_kmh'] == 40.0, '临时限速应压过固定限速'
assert out3['lane_cmd'] == 'keep_left', '右车道可能封闭 -> 向左'
assert out3 == SCENE['gt'], 'VLA 路线应与人类司机一致'
print()
print('✅ 两项都对。而且对的原因是**可组合的语义先验**，不是"施工改道"这条规则：')
print('   换成没见过的临时标志、换个国家，P1/P3/P4 仍然成立。')
print('⚠️  代价见下一节：延迟、以及"它也可能把错误输入合理化"。')

## 6 · 三代架构的量化对比

同一个场景、同一份感知输出，把三条路线的**信息流与代价**摆在一起。
延迟数字用的是各路线的典型量级（车端），不是精确测量。

In [ ]:
ARCHS = [
    # 名称,        输出,   可检查中间量数, 典型延迟ms, 新长尾场景的修复动作, 修复成本
    ('① 模块化',   out1, len([c for v in trace1.values() for c in v]), 45,
     '写一条新规则', '小时级，但会与已有规则冲突'),
    ('② 端到端',   out2, 0, 90,
     '采集+标注一批数据重训', '周级'),
    ('③ VLA',      out3, len(dbg3['reasoning']), 600,
     '改一句 prompt 先验 / 或蒸馏一批数据', '分钟级（prompt）到天级（蒸馏）'),
]

print(f"{'架构':<10s}{'v_target':>10s}{'lane_cmd':>12s}{'正确':>6s}"
      f"{'中间量':>8s}{'延迟ms':>8s}")
for name, out, n_mid, lat, _, _ in ARCHS:
    ok = '✅' if out == SCENE['gt'] else '❌'
    v = 'None' if out['v_target_kmh'] is None else f"{out['v_target_kmh']:.0f}"
    print(f'{name:<10s}{v:>10s}{out["lane_cmd"]:>12s}{ok:>6s}{n_mid:>8d}{lat:>8d}')

print()
print(f"{'架构':<10s}{'新长尾场景怎么修':<30s}{'修复成本'}")
for name, _, _, _, fix, cost in ARCHS:
    print(f'{name:<10s}{fix:<30s}{cost}')

correct = [name for name, out, *_ in ARCHS if out == SCENE['gt']]
assert correct == ['③ VLA'], f'只有 VLA 路线应答对，实际 {correct}'
assert ARCHS[1][2] == 0, '端到端没有可检查的中间量'
assert ARCHS[2][3] > 10 * ARCHS[0][3], 'VLA 延迟比模块化高一个数量级以上'
print()
print('⚠️  别只看"正确"那一列。VLA 的 600 ms 延迟意味着它**不可能**直接控车：')
print('    量产做法是 VLA 出粗轨迹/意图（几 Hz），下游控制器高频跟踪（10-100 Hz）。')
print('✅ 三代不是替代关系。量产架构 = 模块化的可测性 + VLA 的语义泛化 + 规则安全层。')

## 7 · 为什么规则一定写不完：Zipf 覆盖率模型

场景类型按频率排序近似服从幂律 $f_r \propto r^{-\alpha}$（$\alpha \approx 1$）。
一条规则覆盖一种场景类型，那么 K 条规则覆盖的场景**实例**比例是
$\mathrm{Cov}(K) = H_K / H_N$，于是 $K(c) \approx N^{c}$。

这个式子的含义很反直觉：**要多覆盖 1 个百分点，规则数是乘上去的而不是加上去的。**

In [ ]:
def zipf_freq(N, alpha=1.0):
    r = np.arange(1, N + 1, dtype=float)
    w = r ** (-alpha)
    return w / w.sum()

def coverage(K, N, alpha=1.0):
    '''K 条规则（覆盖最常见的 K 种场景）能覆盖的场景实例比例。'''
    f = zipf_freq(N, alpha)
    return float(f[:K].sum())

def rules_needed(c, N, alpha=1.0):
    '''覆盖 c 比例的场景实例，最少需要多少条规则。'''
    f = zipf_freq(N, alpha)
    cum = np.cumsum(f)
    return int(np.searchsorted(cum, c) + 1)

N = 5000                       # 一个国家路网上的场景类型数（保守估计）
print(f'场景类型数 N = {N}, alpha = 1.0')
print()
print(f"{'目标覆盖率':>10s}{'需要规则数 K':>14s}{'相比上一档新增':>16s}{'每条新规则的边际覆盖':>22s}")
prev_k = 0
rows = []
for c in [0.50, 0.90, 0.99, 0.999]:
    k = rules_needed(c, N)
    marginal = (c - coverage(prev_k, N)) / max(k - prev_k, 1)
    rows.append((c, k, k - prev_k, marginal))
    print(f'{c:>10.1%}{k:>14d}{k - prev_k:>16d}{marginal:>21.5%}')
    prev_k = k

k50, k90, k99 = rules_needed(0.5, N), rules_needed(0.9, N), rules_needed(0.99, N)
assert 45 <= k50 <= 60, f'50% 覆盖约需 53 条，得到 {k50}'
assert 1900 <= k90 <= 2100, f'90% 覆盖约需 2014 条，得到 {k90}'
assert 4400 <= k99 <= 4700, f'99% 覆盖约需 4562 条，得到 {k99}'
assert (k99 - k90) > (k90 - k50) / 2, '最后 9% 的成本必须与前面 40% 可比'

print()
print(f'✅ **{k50} 条规则覆盖一半的路况；最后 9% 要再写 {k99 - k90} 条。**')
print(f'   而且这 {k99 - k90} 条彼此还会冲突，维护成本要再乘一个系数。')

# 与解析近似 K(c) ≈ N^c 对照
for c in [0.5, 0.9, 0.99]:
    print(f'  c={c:.2f}: 数值解 {rules_needed(c, N):>5d}   解析近似 N^c = {N ** c:>8.0f}')
print()
print('⚠️  alpha 越小（尾巴越肥），情况越糟 —— 下面验证：')
for a in [1.2, 1.0, 0.8]:
    print(f'  alpha={a:.1f}: 覆盖 99% 需要 {rules_needed(0.99, N, a):>5d} 条规则')
assert rules_needed(0.99, N, 0.8) > rules_needed(0.99, N, 1.2), '尾巴越肥，规则越多'
print('✅ 这就是"规则法成本随覆盖率超线性上升"的确切含义。')

## ✏️ 练习 1：覆盖率与规则预算

实现 `rule_budget(target_cov, N, alpha, hours_per_rule)`，返回
`{'K': 规则数, 'cov': 实际覆盖率, 'person_hours': 总工时, 'marginal_hours_per_pct': 最后一个百分点的工时}`。

- `K` 用「累积频率首次 ≥ target_cov 的下标 + 1」
- `marginal_hours_per_pct` = 从 `target_cov - 0.01` 覆盖率提升到 `target_cov` 所需的**新增规则数** × `hours_per_rule`

In [ ]:
def rule_budget(target_cov, N=5000, alpha=1.0, hours_per_rule=4.0):
    # TODO:
    #  ① 用 zipf_freq + cumsum 求 K
    #  ② cov = 前 K 项之和
    #  ③ person_hours = K * hours_per_rule
    #  ④ marginal = (K(target) - K(target-0.01)) * hours_per_rule
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
b50 = rule_budget(0.50)
b99 = rule_budget(0.99)
assert 45 <= b50['K'] <= 60, b50
assert 4400 <= b99['K'] <= 4700, b99
assert b50['cov'] >= 0.50 and b99['cov'] >= 0.99
assert abs(b99['person_hours'] - b99['K'] * 4.0) < 1e-6
assert b99['marginal_hours_per_pct'] > 30 * b50['marginal_hours_per_pct'], \
    '最后一个百分点的边际成本应比中段贵一个数量级以上'
print(f"{'目标覆盖':>10s}{'K':>8s}{'总工时':>10s}{'最后1%的工时':>16s}")
for c in [0.50, 0.80, 0.90, 0.95, 0.99]:
    b = rule_budget(c)
    print(f'{c:>10.0%}{b["K"]:>8d}{b["person_hours"]:>10.0f}{b["marginal_hours_per_pct"]:>16.0f}')
print()
print('✅ 练习 1 通过：**规则法的成本曲线是超线性的，这是换范式的经济学理由**')

## ✏️ 练习 2：找出信息在流水线的哪一步丢失

实现 `information_loss(scene, rules, score_th)`，返回一个 dict：
每个检测的 `cls` → 它**在哪一步被丢弃**，取值为
`'kept'` / `'lowscore'` / `'wrong_lane'` / `'unknown_class'`。

判定顺序必须与 `modular_plan` 一致：先分数、再车道、再类别表。

In [ ]:
def information_loss(scene, rules=RULES, score_th=SCORE_TH):
    # TODO: 对 scene['dets'] 里每个检测，按 分数 -> 车道 -> 类别表 的顺序判定归宿
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
loss = information_loss(SCENE)
assert loss['speed_limit_80'] == 'kept'
assert loss['speed_limit_40'] == 'kept'
assert loss['construction_ahead'] == 'unknown_class', loss
assert loss['lane_closed_right'] == 'lowscore', '0.44 < 0.60，先被分数门限拦下'
assert set(loss) == {d['cls'] for d in SCENE['dets']}
# 顺序敏感性：把阈值降到 0.4，lane_closed_right 就改为在车道那一步被丢
loss2 = information_loss(SCENE, score_th=0.40)
assert loss2['lane_closed_right'] == 'wrong_lane', loss2
print(f"{'类别':<22s}{'归宿'}")
for k, v in loss.items():
    mark = '✅' if v == 'kept' else '❌'
    print(f'{k:<22s}{mark} {v}')
print()
print('✅ 练习 2 通过：**模块化的每一处信息丢失都是"符合设计"的** ——')
print('   所以事故复盘时你找不到 bug，只能找到"当初没想到"。')

## ✏️ 练习 3：分层控制的频率账

VLA 推理只有几 Hz，而车辆控制需要 10–100 Hz。量产解法是**分层**：
VLA 输出一段未来 `chunk_k` 步的动作（动作分块，m02 详述），下游控制器以 `ctrl_hz` 高频执行。

实现 `layered_ok(vla_hz, chunk_k, ctrl_hz)`，返回
`{'vla_period_s':…, 'chunk_span_s':…, 'covered':bool, 'gap_s':…}`。

- `vla_period_s = 1 / vla_hz` —— 两次 VLA 推理之间的间隔
- `chunk_span_s = chunk_k / ctrl_hz` —— 一个动作块能撑多久
- `covered` = `chunk_span_s >= vla_period_s`（撑得住才不会出现"没有动作可执行"的空窗）
- `gap_s = max(0, vla_period_s - chunk_span_s)`

In [ ]:
def layered_ok(vla_hz, chunk_k, ctrl_hz):
    # TODO
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
r = layered_ok(vla_hz=2.0, chunk_k=50, ctrl_hz=50.0)
assert abs(r['vla_period_s'] - 0.5) < 1e-9
assert abs(r['chunk_span_s'] - 1.0) < 1e-9
assert r['covered'] and abs(r['gap_s']) < 1e-9
bad = layered_ok(vla_hz=2.0, chunk_k=10, ctrl_hz=50.0)
assert not bad['covered'] and abs(bad['gap_s'] - 0.3) < 1e-9, bad
print(f"{'VLA Hz':>8s}{'chunk_k':>9s}{'ctrl Hz':>9s}{'块时长s':>10s}{'周期s':>9s}{'空窗s':>9s}{'OK':>5s}")
for vla_hz, k, ctrl in [(1.0, 20, 50), (2.0, 50, 50), (2.0, 10, 50), (0.5, 100, 50), (5.0, 20, 100)]:
    r = layered_ok(vla_hz, k, ctrl)
    print(f'{vla_hz:>8.1f}{k:>9d}{ctrl:>9.0f}{r["chunk_span_s"]:>10.2f}'
          f'{r["vla_period_s"]:>9.2f}{r["gap_s"]:>9.2f}{"✅" if r["covered"] else "❌":>5s}')
print()
print('✅ 练习 3 通过：**"VLA 延迟几百 ms 怎么可能上车" 的答案就是这张表** ——')
print('   不是让 VLA 变快，而是让它一次输出足够长的动作块。')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def rule_budget(target_cov, N=5000, alpha=1.0, hours_per_rule=4.0):
    f = zipf_freq(N, alpha)
    cum = np.cumsum(f)
    K = int(np.searchsorted(cum, target_cov) + 1)
    K_prev = int(np.searchsorted(cum, max(target_cov - 0.01, 0.0)) + 1)
    return {'K': K,
            'cov': float(cum[K - 1]),
            'person_hours': K * hours_per_rule,
            'marginal_hours_per_pct': (K - K_prev) * hours_per_rule}

In [ ]:
# 练习 2 参考答案
def information_loss(scene, rules=RULES, score_th=SCORE_TH):
    out = {}
    for d in scene['dets']:
        if d['score'] < score_th:
            out[d['cls']] = 'lowscore'
        elif d['lane'] != 'ego':
            out[d['cls']] = 'wrong_lane'
        elif d['cls'] not in rules:
            out[d['cls']] = 'unknown_class'
        else:
            out[d['cls']] = 'kept'
    return out

In [ ]:
# 练习 3 参考答案
def layered_ok(vla_hz, chunk_k, ctrl_hz):
    vla_period_s = 1.0 / vla_hz
    chunk_span_s = chunk_k / ctrl_hz
    return {'vla_period_s': vla_period_s,
            'chunk_span_s': chunk_span_s,
            'covered': chunk_span_s >= vla_period_s,
            'gap_s': max(0.0, vla_period_s - chunk_span_s)}

---
## 🧪 真实工程胶囊：TSR → VLA 接口契约草案

这份契约是 m03 的正式内容，这里先给出骨架——**建议现在就读一遍字段注释**，
后面每一节都会回来给某个字段补上「为什么必须有」。

In [ ]:
RECIPE = r'''
# ── configs/interfaces/tsr_to_vla.yaml ──────────────────────────────
# 随模型一起版本化。任何字段变更都要升 schema_version 并跑接口回归。
schema_version: "0.1"

frame:
  ts_ns: int                 # 感知帧时间戳。**必须是传感器时间，不是系统时间**
  ego_speed_mps: float       # VLA 需要它才能判断"80m 外的牌还来不来得及反应"
  ego_lane_id: str

signs:                       # 每个元素 = 一个被**跟踪**的标志实例（不是逐帧检测框）
  - track_id: int            # 有 track_id 才能表达"首次检出/已确认 N 帧"
    cls: str                 # 层次类别: regulatory.speed_limit / warning.construction
    value: float | null      # 数值语义（限速值）；非数值类为 null，不要塞进 cls 字符串
    score: float             # ** 必须传。不传 = 强迫下游把所有检测当真 **
    score_calibrated: bool   # 未校准的 0.9 不等于 90% —— 下游必须知道这件事
    dist_m: float
    lateral_offset_m: float
    lane_assoc: enum[ego, left, right, oncoming, unknown]   # unknown 必须可表达
    source: enum[perception, hdmap, v2x]
    validity: enum[permanent, temporary, conditional]       # 临时 vs 固定的优先级依据
    first_seen_ts_ns: int
    n_frames_confirmed: int  # "看到了" 与 "确认了" 是两件事

policy_hints:                # 交给 VLA 的自然语言先验（与 schema 一起版本化）
  - "临时标志优先于固定标志。"
  - "score < 0.6 的信息不可直接执行，但可作为保守化的理由。"
  - "lane_assoc=unknown 时，按最保守的车道假设处理。"

# ── 接口回归的三条最低门禁 ──────────────────────────────────────
# 1. 字段完整性：signs[*].score 缺失 -> 直接 reject，不允许默认 1.0
# 2. 顺序稳定性：signs 按 dist_m 升序；同距离按 track_id 升序（prompt 才可复现）
# 3. 降级路径：感知超时 -> 发空 signs + degraded=true，而不是发上一帧的旧结果
'''
print(RECIPE)
for key in ['score_calibrated', 'track_id', 'lane_assoc', 'validity',
            'n_frames_confirmed', 'policy_hints', 'degraded=true']:
    assert key in RECIPE, key
print()
print('✅ 契约覆盖：置信度传递 / 跟踪身份 / 车道关联 / 临时优先 / 确认状态 / 降级路径')
print('   —— 这七项里任何一项缺失，都对应一类真实的路测事故。m03 逐项展开。')

### 小结

- **三代架构不是替代关系，是权衡的三个不同站位**：模块化拿信息完整性换可定位性；
  端到端换回来；VLA 用语言当中间表示，试图两头都要一点。**量产是三者的混合体。**
- **VLA 的定义里最重要的词是「预训练」**。从头训一个吃图吐轨迹的网络是端到端，不是 VLA。
  它的全部价值来自继承的视觉-语言常识。
- **核心痛点是长尾语义理解，而长尾的形状是可以算出来的**：Zipf 分布下
  **53 条规则覆盖 50%，最后 9% 要再写 2548 条**。规则法成本超线性上升，
  这就是换范式的经济学理由。
- **能力边界必须能脱口而出**：延迟（几 Hz vs 控制需要的 10–100 Hz）、幻觉、
  长尾仍是长尾、不可验证。**面试里主动划边界是加分项，不是减分项。**
- **「几百 ms 延迟怎么上车」的答案是分层 + 动作分块**，不是让模型变快。
- **本课的战场是感知与 VLA 之间的接口**：传什么字段、用什么形式传、
  置信度怎么传、感知错了怎么让下游保守。**「只传类别不传置信度」是这个接口最常见的设计错误。**

下一站：**模块 01 · 从 VLM 到 VLA** —— 动作是怎么变成 token 的，
以及机器人 VLA 与自驾 VLA 到底差在哪。